# IPA Transcriptor on Colab

Clone the repository into the runtime.

In [ ]:
%%bash
git clone https://github.com/pymlex/ipa-transcriptor-300M.git
cd ipa-transcriptor-300M

Install Python dependencies from `requirements.txt`.

In [ ]:
%%bash
cd ipa-transcriptor-300M
pip install -q -r requirements.txt

Create `.env` with Kaggle and Hugging Face tokens.

In [ ]:
%%bash
cd ipa-transcriptor-300M
cp .env.example .env
echo "Edit .env in the file browser before the next steps."

Download the Kaggle dictionary archive.

In [ ]:
%%bash
cd ipa-transcriptor-300M
export PYTHONPATH=.
bash scripts/download.sh

Build train, validation, and test CSV splits.

In [ ]:
%%bash
cd ipa-transcriptor-300M
export PYTHONPATH=.
bash scripts/prepare.sh

Fine-tune ByT5. Ends with `runs/<RUN_NAME>/best`.

In [ ]:
%%bash
cd ipa-transcriptor-300M
git pull
export PYTHONPATH=.
export PYTHONUNBUFFERED=1
export RUN_NAME=colab_l4_bf16
bash scripts/train.sh

Benchmark on val and test. Writes `runs/colab_l4_bf16/benchmark.json`.

In [ ]:
%%bash
cd ipa-transcriptor-300M
export PYTHONPATH=.
export CHECKPOINT=runs/colab_l4_bf16/best
export OUTPUT=runs/colab_l4_bf16/benchmark.json
bash scripts/evaluate.sh

Upload weights to Hugging Face Hub.

In [ ]:
%%bash
cd ipa-transcriptor-300M
export PYTHONPATH=.
export CHECKPOINT=runs/colab_l4_bf16/best
export HF_REPO_ID=pymlex/ipa-transcriptor-300M
bash scripts/push_hf.sh

Push metrics to GitHub. Weights stay on the Hub, not in git.

In [ ]:
%%bash
cd ipa-transcriptor-300M
git pull
if [ -f benchmark.json ] && [ ! -f runs/colab_l4_bf16/benchmark.json ]; then
  mv benchmark.json runs/colab_l4_bf16/benchmark.json
fi
git add runs/colab_l4_bf16/metrics.csv runs/colab_l4_bf16/benchmark.json
git commit -m "Colab L4 ByT5 fine-tuning run"
git push